In [0]:
test_df = fetch_ticker_history_range("AAPL", "2024-01-01", "2024-06-01")
test_df.head()

In [0]:
raw_pdf["ticker"].value_counts()
raw_pdf[raw_pdf["ticker"] == "COST"]["date"].agg(["min", "max", "count"])

raw_pdf[raw_pdf["ticker"] == "AAPL"]["date"].agg(["min", "max", "count"])
aapl_dates = set(raw_pdf[raw_pdf["ticker"] == "AAPL"]["date"])
cost_dates = set(raw_pdf[raw_pdf["ticker"] == "COST"]["date"])
aapl_dates - cost_dates

In [0]:
%sql
SHOW CATALOGS

In [0]:
%sql
SHOW TABLES IN mashup_learning.stocks


In [0]:
%sql
DESCRIBE TABLE mashup_learning.stocks.bronze_daily_prices;


In [0]:
%sql
SELECT COUNT(*) FROM mashup_learning.stocks.bronze_daily_prices;
    


In [0]:
%sql DESCRIBE HISTORY mashup_learning.stocks.bronze_daily_prices

In [0]:
%sql
SELECT 
CAST(date AS DATE) as trade_date,
LAG(close, 1, 0) OVER (PARTITION BY ticker ORDER BY date) AS previous_day_close,
case when previous_day_close = 0 then 0 
else (close-previous_day_close)/previous_day_close*100 end AS daily_return,
AVG(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS rolling_avg_7d,
open,  high, low, close, volume, dividends, stock_splits, ticker, source, ingested_at
FROM  mashup_learning.stocks.bronze_daily_prices;
--WHERE TICKER = 'NVDA'
--ORDER BY DATE ASC;

In [0]:
%sql
SELECT * FROM (SELECT 
CAST(date AS DATE) as trade_date,
LAG(close, 1, NULL) OVER (PARTITION BY ticker ORDER BY date) AS previous_day_close,
case when previous_day_close is null then null 
else (close-previous_day_close)/previous_day_close*100 end AS daily_return,
case WHEN
COUNT(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)<7 THEN NULL 
ELSE AVG(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)  end AS rolling_avg_7d,
open,  high, low, close, volume, dividends, stock_splits, ticker, source, ingested_at
FROM  mashup_learning.stocks.bronze_daily_prices)
WHERE ticker = 'AAPL'
ORDER BY "date"
LIMIT 10;

In [0]:
%sql
DESCRIBE HISTORY mashup_learning.stocks.silver_daily_prices;